# NeuralGCM Atmospheric River Predictions CLI

In [1]:
import gcsfs
import jax
import numpy as np
import pickle
import xarray as xr
from pathlib import Path

from dinosaur import horizontal_interpolation
from dinosaur import spherical_harmonic
from dinosaur import xarray_utils
import neuralgcm

from prefect import task, flow
from loguru import logger
import sys
import typer

## Tasks Definitions

In [ ]:
@task
def load_model(model_name: str = 'v1/deterministic_2_8_deg.pkl', token: str = 'anon') -> neuralgcm.PressureLevelModel:
    fs = gcsfs.GCSFileSystem(token=token)
    with fs.open(f'gs://neuralgcm/models/{model_name}', 'rb') as f:
        ckpt = pickle.load(f)
    return neuralgcm.PressureLevelModel.from_checkpoint(ckpt)

In [ ]:
@task
def load_era5(era5_path: str, token: str = 'anon') -> xr.Dataset:
    return xr.open_zarr(era5_path, chunks=None, storage_options=dict(token=token))

In [ ]:
@task
def preprocess_era5(full_era5: xr.Dataset, model, start_time: str, end_time: str, data_inner_steps: int) -> xr.Dataset:
    sliced = (
        full_era5[model.input_variables + model.forcing_variables]
        .pipe(xarray_utils.selective_temporal_shift, variables=model.forcing_variables, time_shift=f'{data_inner_steps} hours')
        .sel(time=slice(start_time, end_time, data_inner_steps))
        .compute()
    )
    era5_grid = spherical_harmonic.Grid(
        latitude_nodes=full_era5.sizes['latitude'],
        longitude_nodes=full_era5.sizes['longitude'],
        latitude_spacing=xarray_utils.infer_latitude_spacing(full_era5.latitude),
        longitude_offset=xarray_utils.infer_longitude_offset(full_era5.longitude),
    )
    regridder = horizontal_interpolation.ConservativeRegridder(era5_grid, model.data_coords.horizontal, skipna=True)
    eval_era5 = xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(sliced, regridder))
    return eval_era5

In [ ]:
@task
def forecast(model, eval_era5: xr.Dataset, init_time: str, hour_inner_steps: int, days: int) -> xr.Dataset:
    init_np = np.datetime64(init_time)
    inner_steps = hour_inner_steps
    outer_steps = days * 24 // inner_steps
    timedelta = np.timedelta64(1, 'h') * inner_steps
    times = np.arange(outer_steps) * inner_steps
    inputs = model.inputs_from_xarray(eval_era5.sel(time=init_np))
    initial_state = model.encode(inputs, model.forcings_from_xarray(eval_era5.sel(time=init_np)), jax.random.key(42))
    final_state, predictions = model.unroll(initial_state, model.forcings_from_xarray(eval_era5.head(time=1)), steps=outer_steps, timedelta=timedelta, start_with_input=True)
    predictions_ds = model.data_to_xarray(predictions, times=times)
    target_traj = model.inputs_from_xarray(eval_era5.thin(time=(inner_steps // hour_inner_steps)).isel(time=slice(outer_steps)))
    target_ds = model.data_to_xarray(target_traj, times=times)
    combined = xr.concat([target_ds, predictions_ds], dim='model')
    combined.coords['model'] = ['ERA5', 'NeuralGCM']
    return combined

In [ ]:
@task
def save_zarr(ds: xr.Dataset, path: Path) -> None:
    ds.to_zarr(path, mode='w', compute=True)

## Main Flow

In [ ]:
@flow
@app.command()
def main(model_name: str, era5_path: str, start_time: str, end_time: str, inner_steps: int, days: int, save: bool):
    model = load_model(model_name)
    full_era5 = load_era5(era5_path)
    eval_era5 = preprocess_era5(full_era5, model, start_time, end_time, inner_steps)
    combined = forecast(model, eval_era5, f'{start_time}T00:00:00', inner_steps, days)
    if save:
        save_zarr(combined)
        logger.success('Forecast complete and saved to Zarr.')
    else:
        logger.success('Forecast complete.')

In [ ]:
import xarray as xr
local_zarr=xr.open_zarr('./neuralgcm_atmospheric_river_predictions.zarr',chunks=None)
xr.testing.assert_allclose(local_zarr,combined_ds,rtol=1e-5,atol=1e-5)

/home/sungche/.conda/envs/neuralgcm/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


<xarray.Dataset> Size: 85MB
Dimensions:                              (model: 2, time: 5, level: 37,
                                          longitude: 128, latitude: 64)
Coordinates:
  * time                                 (time) int64 40B 0 24 48 72 96
  * longitude                            (longitude) float64 1kB 0.0 ... 357.2
  * model                                (model) object 16B 'ERA5' 'NeuralGCM'
  * level                                (level) int64 296B 1 2 3 ... 975 1000
  * latitude                             (latitude) float64 512B -87.86 ... 8...
Data variables:
    specific_humidity                    (model, time, level, longitude, latitude) float32 12MB ...
    specific_cloud_liquid_water_content  (model, time, level, longitude, latitude) float32 12MB ...
    v_component_of_wind                  (model, time, level, longitude, latitude) float32 12MB ...
    temperature                          (model, time, level, longitude, latitude) float32 12MB ...
    geopotential                         (model, time, level, longitude, latitude) float32 12MB ...
    u_component_of_wind                  (model, time, level, longitude, latitude) float32 12MB ...
    specific_cloud_ice_water_content     (model, time, level, longitude, latitude) float32 12MB ...
    sim_time                             (model, time) float32 40B ...
Attributes:
    longitude_wavenumbers:     64
    total_wavenumbers:         65
    longitude_nodes:           128
    latitude_nodes:            64
    latitude_spacing:          gauss
    longitude_offset:          0.0
    radius:                    1.0
    spherical_harmonics_impl:  RealSphericalHarmonics
    spmd_mesh:                 
    centers:                   [1, 2, 3, 5, 7, 10, 20, 30, 50, 70, 100, 125, ...
    horizontal_grid_type:      Grid
    vertical_grid_type:        PressureCoordinates